# 🧠 ArquiSys AI - Ollama en Google Colab

Este notebook ejecuta Ollama con qwen2.5:3b y expone el endpoint via ngrok para usar desde tu sistema local.

## Instrucciones:
1. Ejecutar todas las celdas en orden
2. Al final you'll get una URL de ngrok (ej: https://abcd1234.ngrok.io)
3. En tu .env local cambiar: `OLLAMA_BASE_URL=https://TU-URL.ngrok.io`
4. Asegurarse que USE_OLLAMA_FOR_ANALYST=true

In [ ]:
#@title 1. Instalar dependencias y Ollama
# Instalar zstd primero
!apt-get update && apt-get install -y zstd
# Luego instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh
import os
os.environ['PATH'] += ':/root/.local/bin'
print("✅ Ollama instalado")

In [ ]:
#@title 2. Instalar ngrok
!wget -q https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip -O ngrok.zip
!unzip -o ngrok.zip
!mv ngrok /usr/local/bin/
print("✅ ngrok instalado")

In [ ]:
#@title 3. Configurar token de ngrok
#@markdown Ingresa tu token de ngrok (gratis en https://dashboard.ngrok.com/get-started)
NGROK_TOKEN = "cr_3D2ftOD9E86bIfdoD8Z3eYEDplc"  #@param {type:"string"}

import os
if NGROK_TOKEN:
    !mkdir -p ~/.ngrok2
    !echo 'authtoken: {NGROK_TOKEN}' > ~/.ngrok2/ngrok.yml
    !ngrok config upgrade
    print("✅ Token de ngrok configurado")
else:
    print("⚠️ Token de ngrok no configurado - podés usar sin-auth pero es menos seguro")

In [ ]:
#@title 4. Instalar Ollama si no existe
import subprocess
import os

ollama_path = "/root/.local/bin/ollama"

if not os.path.exists(ollama_path):
    print("⚠️ Ollama no encontrado, instalando...")
    !apt-get update && apt-get install -y zstd
    !curl -fsSL https://ollama.com/install.sh | sh
    os.environ['PATH'] += ':/root/.local/bin'
    print("✅ Ollama instalado")
else:
    print("✅ Ollama ya instalado")

os.environ['PATH'] += ':/root/.local/bin'

In [ ]:
#@title 5. Iniciar Ollama en background
import subprocess
import time

ollama_path = "/root/.local/bin/ollama"

ollama_process = subprocess.Popen(
    [ollama_path, "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    start_new_session=True
)
time.sleep(5)
print("✅ Ollama iniciado en puerto 11434")

In [ ]:
#@title 6. Descargar modelo qwen2.5:3b
#@markdown Esto descarga ~2GB - puede tomar unos minutos
import subprocess
import os

ollama_path = "/root/.local/bin/ollama"

# Verificar si el modelo ya está descargado
result = subprocess.run([ollama_path, "list"], capture_output=True, text=True)
if "qwen2.5:3b" in result.stdout:
    print("✅ Modelo qwen2.5:3b ya está descargado")
else:
    print("📥 Descargando modelo qwen2.5:3b (puede tomar unos minutos)...")
    result = subprocess.run([ollama_path, "pull", "qwen2.5:3b"], 
                           capture_output=True, text=True, timeout=600)
    if result.returncode == 0:
        print("✅ Modelo qwen2.5:3b descargado")
    else:
        print("⚠️ Descarga puede haber fallado")
        print(result.stderr)

In [ ]:
#@title 7. Iniciar ngrok y mostrar URL
import subprocess
import time

# Iniciar ngrok
ngrok_process = subprocess.Popen(
    ["/usr/local/bin/ngrok", "http", "11434", "--host-header", "rewrite"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    start_new_session=True
)

time.sleep(10)

# Obtener URL de ngrok
try:
    import urllib.request
    response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
    data = response.read().decode()
    import json
    tunnels = json.loads(data)['tunnels']
    public_url = tunnels[0]['public_url']
    print(f"🎉 URL PÚBLICA DE OLLAMA:")
    print(f"   {public_url}")
    print(f"\n📝 Para usar en tu sistema local, configura:")
    print(f"   OLLAMA_BASE_URL={public_url}")
    print(f"\n⚠️ Esta URL cambia cada vez que reinicies el notebook")
except Exception as e:
    print(f"⚠️ Error obteniendo URL: {e}")
    print("Esperando más tiempo...")
    time.sleep(15)
    try:
        response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
        data = response.read().decode()
        import json
        tunnels = json.loads(data)['tunnels']
        public_url = tunnels[0]['public_url']
        print(f"🎉 URL PÚBLICA: {public_url}")
    except:
        print("❌ No se pudo obtener la URL")

In [ ]:
#@title 8. Verificar que funciona
import urllib.request
import json

# Probar que Ollama responde
try:
    test_url = "http://localhost:11434/api/tags"
    if 'public_url' in globals():
        test_url = f"{public_url}/api/tags"
    
    response = urllib.request.urlopen(test_url, timeout=15)
    models = json.loads(response.read())
    print("✅ Modelos disponibles:")
    for m in models.get('models', []):
        print(f"   - {m['name']}")
    
    print("\n🎉 TODO LISTO! Ollama está funcionando en la nube.")
except Exception as e:
    print(f"⚠️ Verificando... {e}")
    print("Puede que el modelo aún se esté descargando. Esperando 30 segundos...")
    import time
    time.sleep(30)
    try:
        response = urllib.request.urlopen("http://localhost:11434/api/tags", timeout=15)
        models = json.loads(response.read())
        print("✅ Modelos disponibles:")
        for m in models.get('models', []):
            print(f"   - {m['name']}")
    except:
        print("❌ Ollama aún no responde. Probá ejecutar la celda de descarga nuevamente.")

## 🔧 Configuración en tu sistema local

Una vez que obtengas la URL de ngrok, actualizá tu archivo `.env`:

```bash
USE_OLLAMA_FOR_ANALYST=true
OLLAMA_MODEL=qwen2.5:3b
OLLAMA_BASE_URL=https://xxxx-xxxx.ngrok.io
```

**Nota:** Cada vez que reinicies el notebook, la URL de ngrok cambia. Deberás actualizar tu .env cada vez.